# rl17_update

> Phase 17, the pieces of `trainRL17`'s step: the step's board groups (`boardGroups`: team sizes from `mixN`, scripted liars on a share of the boards), the reply-level clipped policy-gradient term (`clippedTerm`), the update passes with the reference and old log-probabilities computed once per step (`ppoUpdate`), and the turn logs of every board group (`saveTurns`). The offline tests need no GPU.

In [ ]:
#| default_exp rl17_update

In [ ]:
#| hide
from nbdev.showdoc import *

## Board groups, the loss, the passes

In [ ]:
#| export
import json
from pathlib import Path

import numpy as np

from explRL.rl_torch import replyLogps


def boardGroups(cfg, s):
    """The step's board groups: [(N, nLiar, seeds)] with each board seed repeated `group` times. The step's distinct seeds
    (as stepSeeds) are split in order by mixN; within a team size the first round(liarShare x boards) carry N // 4 liars."""
    distinct = [cfg.boardSeed0 + s * cfg.seedsPerStep + j for j in range(cfg.seedsPerStep)]
    out, at = [], 0
    for k, (N, share) in enumerate(cfg.mixN):
        n = len(distinct) - at if k == len(cfg.mixN) - 1 else int(round(cfg.seedsPerStep * share))
        mine, at = distinct[at:at + n], at + n
        nl = int(round(cfg.liarShare * len(mine)))
        for liars, seeds in ((N // 4, mine[:nl]), (0, mine[nl:])):
            if seeds:
                out.append((N, liars, [x for x in seeds for _ in range(cfg.group)]))
    return out


def clippedTerm(lp, old, adv, clip):
    "One reply's policy-gradient term: Phase 12's -A * mean log pi without a clip, else PPO's clipped reply-level ratio."
    import torch
    if clip is None:
        return -adv * lp.mean()
    r = torch.exp(lp.mean() - old.mean())
    return -torch.min(r * adv, torch.clamp(r, 1 - clip, 1 + clip) * adv)


def ppoUpdate(model, tok, opt, params, samples, chunks, kl, maxLen, device, epochs=1, clip=None, seed=0):
    """`epochs` passes over `samples` (micro-batches `chunks`; passes after the first in a seeded shuffled order); the
    reference (adapter off) and old log-probabilities once, before the first pass; per pass: the clipped term plus
    kl x k3 to the reference, gradient clipping 1.0, one optimizer step. -> loss, KL, tokens (the last pass's)."""
    import torch
    with torch.no_grad():
        ref = [replyLogps(model, tok, ch, maxLen, device) for ch in _off(model, chunks)]
        old = [[x.detach() for x in replyLogps(model, tok, ch, maxLen, device)] for ch in chunks] if clip is not None else None
    order = list(range(len(chunks)))
    for e in range(epochs):
        if e:
            np.random.default_rng([seed, e]).shuffle(order)
        stats = dict(loss=0.0, kl=0.0, tokens=0)
        for c in order:
            lps, loss = replyLogps(model, tok, chunks[c], maxLen, device), 0.0
            for n, (x, lp, rf) in enumerate(zip(chunks[c], lps, ref[c])):
                d = rf - lp
                k3 = (torch.exp(d) - d - 1).mean()
                loss = loss + clippedTerm(lp, old[c][n] if old else None, x['adv'], clip) + kl * k3
                stats['kl'] += k3.item() / len(samples)
                stats['tokens'] += len(lp)
            (loss / len(samples)).backward()
            stats['loss'] += loss.item() / len(samples)
        if samples:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        opt.zero_grad()
    return stats


def saveTurns(out, step, gs):
    "out/turns/step-<step>.jsonl: every board group's turn logs (team size and liars on each line)."
    (Path(out) / 'turns').mkdir(exist_ok=True)
    (Path(out) / 'turns' / f'step-{step:04d}.jsonl').write_text(''.join(
        json.dumps(dict(board=b, seed=ro.seeds[b], N=c.N, nLiar=c.nLiar, **x), sort_keys=True) + '\n'
        for c, ro in gs for b in range(len(ro.boards)) for lg in ro.logs[b] for x in lg))


def _off(model, chunks):
    "Yield the chunks with the model's adapter switched off (the warm start: the reference)."
    for ch in chunks:
        with model.disable_adapter():
            yield ch

## Tests

In [ ]:
import numpy as np
from explRL.rl17_update import *
from explRL.rl17_grid import RL17Config
from explRL.clue_rl import stepSeeds

# board groups: the defaults are stepSeeds; the mix splits seeds in order and puts liars on the first board of each size
c0 = RL17Config('m', 'o')
assert boardGroups(c0, 3) == [(4, 0, stepSeeds(c0, 3))]
cm = RL17Config('m', 'o', mixN=((4, 0.5), (8, 0.5)), liarShare=0.25)
g = boardGroups(cm, 0)
assert [(N, nl, len(sd)) for N, nl, sd in g] == [(4, 1, 4), (4, 0, 12), (8, 2, 4), (8, 0, 12)]
assert sorted(set(x for _, _, sd in g for x in sd)) == sorted(set(stepSeeds(cm, 0)))      # the same board seeds as stepSeeds
assert all(sd == [x for x in sd[::cm.group] for _ in range(cm.group)] for _, _, sd in g)              # each board played group times
# liar slots are never trained: on an 8-agent board with 2 scripted liars, every sample comes from an honest slot
from explRL.clue_agents import ClueLiar
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_policy import scriptedPolicy
from explRL.clue_roll import rollout
from explRL.llm_runs import LLMConfig
from explRL.rl_adv import stepSamples
lc = LLMConfig(task='clue', channel='blackboard', N=8, nLiar=2, turns=3, model='m', world={k: CLUE_SETTINGS[k] for k in BOARD})
ro = rollout(lc, [5, 5], scriptedPolicy(lambda s, b: '{"action": "wait", "target": "", "answer": 0, "clue": "", "notes": ""}'),
             {0: ClueLiar, 1: ClueLiar}, keepPrompts=True)
assert all(len(ro.logs[b][i]) == 0 for b in range(2) for i in (0, 1))
samp = stepSamples(ro, np.ones((2, 8)))
assert samp and all(x['slot'] >= 2 for x in samp)

In [ ]:
# the loss terms on hand-made numbers (torch only)
try:
    import torch
    lp, old = torch.tensor([-1.0, -2.0]), torch.tensor([-1.5, -2.5])          # mean ratio exp(0.5)
    assert torch.isclose(clippedTerm(lp, None, 2.0, None), torch.tensor(3.0))   # -A * mean log pi
    r = np.exp(0.5)
    assert torch.isclose(clippedTerm(lp, old, 2.0, 0.2), torch.tensor(-1.2 * 2.0))           # A > 0: clipped at 1 + eps
    assert torch.isclose(clippedTerm(lp, old, -2.0, 0.2), torch.tensor(float(r * 2.0)))      # A < 0: the unclipped ratio
    assert torch.isclose(clippedTerm(lp, lp, 2.0, 0.2), torch.tensor(-2.0))                  # ratio 1
except ImportError:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()